# 01 — Data Audit & EDA

One place for all exploratory analysis: numeric audit, target structure,
train↔validation shift, spatial geometry, WAP correlation matrices,
per-class fingerprints, and predicted-confusion structure.

Every plot/table here supports a downstream modelling decision.

> **`LONGITUDE` / `LATITUDE` are used in §8–9 for visualisation only.**
> They stay on `LEAKAGE_COLUMNS` and never enter any model — see the
> assertion in the imports cell below.

In [ ]:
# --- bootstrap: make src/ importable when notebook started outside `uv run` ---
import sys
from pathlib import Path

_HERE = Path.cwd()
for parent in [_HERE, *_HERE.parents]:
    if (parent / "src" / "ujiindoorloc").is_dir():
        if str(parent / "src") not in sys.path:
            sys.path.insert(0, str(parent / "src"))
        break

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

In [ ]:
import numpy as np
import pandas as pd

from ujiindoorloc.constants import (
    COMBINED_TARGET, FIGURES_DIR, TABLES_DIR, LEAKAGE_COLUMNS,
)
from ujiindoorloc.data_loading import (
    create_building_floor_target,
    get_wap_columns,
    load_raw_data,
    split_features_targets,
)
from ujiindoorloc.preprocessing import (
    get_non_constant_columns, replace_missing_signal_values,
)
from ujiindoorloc import eda
from ujiindoorloc import plots as p
from ujiindoorloc.reporting import ensure_report_dirs, save_table, save_figure_path

# Leakage guard — LON/LAT used below for plotting must remain banned as predictors.
assert "LONGITUDE" in LEAKAGE_COLUMNS and "LATITUDE" in LEAKAGE_COLUMNS

ensure_report_dirs()
raw = load_raw_data()
train, valid = raw.train, raw.valid

# Per-row "# WAPs detected" — handy as a colour for the GPS scatter in §9.
train = train.assign(detected_waps=eda.detected_per_row(train).values)
valid = valid.assign(detected_waps=eda.detected_per_row(valid).values)

print("Train:", train.shape, "  Validation:", valid.shape)

## 1. Basic structure

In [ ]:
shape_df = eda.summarize_dataset_shape(train, valid)
save_table(shape_df, "dataset_shape.csv")
shape_df

In [ ]:
print("First 5 WAP columns :", get_wap_columns(train)[:5])
print("Last 5 WAP columns  :", get_wap_columns(train)[-5:])
print("Non-WAP columns     :", [c for c in train.columns if not c.startswith("WAP")])

## 2. Encoded missingness — the `100` sentinel

Officially the dataset has no NaNs. But the integer `100` in any WAP cell
means *"WAP was not detected during this scan"*. Real RSSI values are
negative. Treating `100` as a strong positive signal would catastrophically
break any distance/scale-based model.

In [ ]:
miss_df = eda.summarize_missingness(train, valid)
save_table(miss_df, "missingness_summary.csv")
miss_df

## 3. WAP signal — per-row detection

In [ ]:
det_train = eda.summarize_wap_detection(train).assign(split="train")
det_valid = eda.summarize_wap_detection(valid).assign(split="validation")
det_df = pd.concat([det_train, det_valid], ignore_index=True)
save_table(det_df, "wap_detection_summary.csv")
det_df

In [ ]:
p.plot_detected_per_row_hist(
    eda.detected_per_row(train),
    save_figure_path("waps_detected_per_row_train.png"),
    title="WAPs detected per scan — training set",
)

## 4. WAP coverage train ↔ validation

In [ ]:
cov_df = eda.summarize_wap_coverage(train, valid)
save_table(cov_df, "wap_coverage_summary.csv")
cov_df

> Some WAPs that appear in validation are **never seen in training**.
> Models cannot learn anything about those, so a validation drop is
> expected — this is a real domain-shift artifact, not a bug.

## 5. Target distributions

In [ ]:
dist_df = eda.summarize_target_distribution(train, valid)
save_table(dist_df, "class_distribution.csv")
dist_df.head(20)

In [ ]:
p.plot_class_distribution(
    train["BUILDINGID"].astype(int),
    save_figure_path("building_distribution.png"),
    "BUILDINGID — training",
    xlabel="BUILDINGID",
)
p.plot_class_distribution(
    train["FLOOR"].astype(int),
    save_figure_path("floor_distribution.png"),
    "FLOOR — training",
    xlabel="FLOOR",
)
p.plot_class_distribution(
    create_building_floor_target(train),
    save_figure_path("building_floor_distribution.png"),
    "building_floor — training (main target)",
    xlabel="building_floor",
)

> Building 2 dominates and floor 4 is small. This is why we will report
> **balanced accuracy** and **macro F1** alongside plain accuracy.

## 6. Metadata shift

The validation split is **not** a random split. It comes from a different
time window, fewer users (1 vs 18), different phones. The numbers below
quantify that shift.

In [ ]:
shift_df = eda.summarize_metadata_shift(train, valid)
save_table(shift_df, "metadata_shift_summary.csv")
shift_df

## 7. WAP correlation analysis

WAPs in the same physical area share signal patterns → high pairwise
correlation → multicollinearity. This motivates regularised models
(logistic regression with L2) and dimensionality reduction (Scenario 5).

In [ ]:
split = split_features_targets(train, valid, target_name=COMBINED_TARGET)
corr_summary, top_pairs = eda.compute_wap_correlations(split.X_train, top_k_pairs=20)
save_table(corr_summary, "wap_correlation_summary.csv")
save_table(top_pairs, "top_correlated_wap_pairs.csv")
corr_summary

In [ ]:
top_pairs.head(20)

In [ ]:
abs_corr_values = eda.correlation_distribution(split.X_train)
p.plot_correlation_distribution(
    abs_corr_values,
    save_figure_path("wap_correlation_distribution.png"),
)

## 8. Campus geometry — buildings & floors

LON/LAT visualised. The first plot proves the buildings are spatially
separated. The 1×3 facet then proves that *within* a building, floors
are **not** geographically separated (a person on floor 0 stands above
the same LON/LAT as one on floor 3) — which is precisely why we need
WiFi to tell them apart.

In [ ]:
bounds = eda.gps_bounds(train, valid)
save_table(bounds, "gps_bounds.csv")
bounds

> Coordinates are projected metres (UTM-like), not lat/lon degrees. Campus footprint ≈ 390 m × 270 m.

In [ ]:
p.plot_gps_scatter(
    train, color_col="BUILDINGID",
    path=save_figure_path("gps_buildings.png"),
    title="Campus footprint — coloured by BUILDINGID",
)
p.plot_gps_facet_by_building(
    train, color_col="FLOOR",
    path=save_figure_path("gps_facet_by_floor.png"),
    title="Per-building floor layout (LON/LAT does NOT separate floors)",
)

## 9. Train ↔ validation spatial overlap & scan density

In [ ]:
p.plot_gps_train_vs_valid(train, valid, save_figure_path("gps_train_vs_valid.png"))

> Validation samples sit *inside* the training footprint — confirming
> that the train↔valid shift documented in §7 is a **device/time/user**
> shift, not a spatial one. That justifies keeping the official split.

In [ ]:
p.plot_gps_density_hexbin(
    train, save_figure_path("gps_density_hexbin.png"),
    title="Training scan density (hexbin)",
)
p.plot_gps_scatter(
    train, color_col="detected_waps",
    path=save_figure_path("gps_detected_waps.png"),
    title="LON/LAT coloured by # WAPs detected per scan (scan quality)",
    sample=8000,
)

## 10. WAP correlation analysis

WAPs in the same physical area share signal patterns → high pairwise
correlation → multicollinearity. This motivates regularised models
(logistic regression with L2) and dimensionality reduction (Scenario 5).

In [ ]:
split = split_features_targets(train, valid, target_name=COMBINED_TARGET)
corr_summary, top_pairs = eda.compute_wap_correlations(split.X_train, top_k_pairs=20)
save_table(corr_summary, "wap_correlation_summary.csv")
save_table(top_pairs, "top_correlated_wap_pairs.csv")
corr_summary

In [ ]:
top_pairs.head(20)

In [ ]:
abs_corr_values = eda.correlation_distribution(split.X_train)
p.plot_correlation_distribution(
    abs_corr_values,
    save_figure_path("wap_correlation_distribution.png"),
)

## 11. Full WAP × WAP correlation heatmap (clustered)

The histogram above told us *how many* pairs are highly correlated.
This is the **matrix** version, reordered by hierarchical clustering so
the AP groups become visually obvious as block structure.

In [ ]:
kept = get_non_constant_columns(split.X_train)
X_clean = replace_missing_signal_values(split.X_train[kept])
corr_full = X_clean.corr()
print("corr matrix:", corr_full.shape)

cluster_order = eda.clustered_corr_order(corr_full)
save_table(
    pd.DataFrame({"order_index": range(len(cluster_order)), "wap": cluster_order}),
    "wap_correlation_clustered_order.csv",
)
p.plot_correlation_heatmap(
    corr_full,
    path=save_figure_path("wap_correlation_heatmap_clustered.png"),
    title=f"WAP × WAP Pearson r — clustered (n={len(corr_full)})",
    cluster_order=cluster_order,
)

## 12. Top-K WAP correlation heatmap (readable subset)

Same matrix, restricted to the 50 most class-discriminative WAPs (top
ANOVA F-scores). Small enough that individual labels are readable.

## 13. Feature relevance to target — ANOVA F-score

In [ ]:
anova_df = eda.compute_anova_feature_scores(split.X_train, split.y_train, top_k=30)
save_table(anova_df, "top_wap_features_anova.csv")
p.plot_top_features(
    anova_df,
    save_figure_path("top_wap_features_anova.png"),
    title="Top 30 WAPs by ANOVA F-score (target=building_floor)",
)
anova_df

In [ ]:
# Top 50 WAPs (used for the next three heatmaps)
anova_top = eda.compute_anova_feature_scores(split.X_train, split.y_train, top_k=50)
top_wap = anova_top["feature"].tolist()
corr_top = corr_full.loc[top_wap, top_wap]
p.plot_correlation_heatmap(
    corr_top,
    path=save_figure_path("wap_correlation_heatmap_top50.png"),
    title="Top 50 WAPs (by ANOVA F) — pairwise Pearson r",
    cluster_order=eda.clustered_corr_order(corr_top),
)

## 14. Class-mean RSSI fingerprint heatmap

Rows = the 13 `building_floor` classes, columns = the top-50 WAPs,
cells = mean RSSI for that class.

> Mean is taken over **all** scans (including not-detected, coded as
> `-110`), so a near-`-110` cell means *"this WAP is rarely heard by
> this class"* — which is what the model actually sees.

In [ ]:
centroids = eda.compute_class_centroids(split.X_train, split.y_train)
centroids_top = centroids[top_wap]
save_table(centroids_top.reset_index(), "class_centroids_top50.csv")
p.plot_class_fingerprint_heatmap(
    centroids_top,
    path=save_figure_path("class_fingerprint_heatmap_top50.png"),
    title="Class-mean RSSI fingerprint — top 50 WAPs (rows = building_floor)",
)

## 15. Per-class WAP detection-rate heatmap

Same axes, but cells are detection rate ∈ [0, 1] — fraction of scans
where the WAP was detected at all. Highlights APs that are
discriminative in *availability*, not just signal strength.

In [ ]:
rates_full = eda.compute_wap_detection_rate_per_class(train)
rates_top = rates_full[top_wap]
save_table(rates_top.reset_index(), "per_class_wap_detection_rate_top50.csv")
p.plot_detection_rate_heatmap(
    rates_top,
    path=save_figure_path("detection_rate_heatmap_top50.png"),
    title="Per-class WAP detection rate — top 50 WAPs",
)

## 16. Class centroid distance heatmap (predicted confusion)

Pairwise Euclidean distance between class centroids in WAP space.
**Small distance ≈ confusable classes** — this matrix predicts the
structure of the confusion matrices we'll see in `nb03`.
Same-building / adjacent-floor pairs typically sit closest together.

In [ ]:
distances = eda.compute_class_centroid_distances(centroids)
save_table(distances.reset_index(), "class_centroid_distances.csv")
p.plot_class_centroid_distance_heatmap(
    distances,
    path=save_figure_path("class_centroid_distance_heatmap.png"),
)

## 17. Per-WAP train ↔ validation drift

One point per WAP. Off-diagonal points are exactly the WAPs whose
behaviour differs between training and validation — they are the most
likely culprits behind any validation accuracy drop.

In [ ]:
shift = eda.compute_per_wap_train_valid_shift(train, valid)
save_table(shift, "per_wap_train_valid_shift.csv")
print("WAPs with no detection in validation:",
      int(shift["valid_mean_rssi"].isna().sum()))
print("WAPs with no detection in training  :",
      int(shift["train_mean_rssi"].isna().sum()))
p.plot_wap_shift_scatter(
    shift, save_figure_path("wap_train_valid_shift_scatter.png"),
)

## 18. Sparsity at a glance

In [ ]:
p.plot_sparsity_heatmap(
    train, save_figure_path("sparsity_heatmap_subsample.png"),
    n_rows=200,
    title="Detection mask — 200-row subsample × all 520 WAPs (black = detected)",
)

## 19. EDA conclusion

* Dataset is **high-dimensional** (520 features) and **sparse** — ~96–97 %
  of WAP cells are the *not-detected* sentinel (§2, §3, §18).
* `100` must be treated as *not detected*, never as a real signal.
* Classes are **imbalanced** — building 2 and floor 0/1 dominate (§5).
* Within a building, **floors are not spatially separated** (§8) — WiFi is
  the only signal that distinguishes them.
* Validation is a **time / device / user shift, not a spatial one** (§9),
  so the official split is kept as the held-out test set.
* Many WAPs are **strongly correlated** with visible block structure (§10–12)
  → use regularisation / PCA / LDA.
* Per-class WAP **fingerprints are visibly distinct** (§14–15), so the
  classification task is genuinely learnable from RSSI alone.
* Centroid-distance heatmap (§16) **predicts** the confusion structure we
  will see in `nb03` — same-building / adjacent-floor pairs sit closest.
* Train↔valid drift (§17) names the specific WAPs likely to hurt
  generalisation — input for any future feature-selection scenario.

These properties make the dataset a clean fit for **Scenario 2**
(parametric vs non-parametric) and **Scenario 5** (DR before
classification).